In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/df_final.csv', parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Shape: (62728, 30)
Columns: ['timestamp', 'price', 'load', 'wind_offshore', 'wind_onshore', 'solar', 'hour', 'day_of_week', 'month', 'temperature', 'wind_speed', 'is_weekend', 'gas_price', 'coal_price', 'price_lag_24h', 'price_lag_168h', 'price_rolling_24h', 'price_rolling_168h', 'co2_price', 'is_holiday', 'is_hol_or_week', 'total_generation', 'net_export', 'coal_generation', 'gas_generation', 'nuclear_generation', 'actual_wind_offshore', 'actual_wind_onshore', 'actual_solar', 'actual_load']


In [2]:
# Checkpoint: verify columns loaded from df_final.csv (should be 30)
df.columns.tolist()

['timestamp',
 'price',
 'load',
 'wind_offshore',
 'wind_onshore',
 'solar',
 'hour',
 'day_of_week',
 'month',
 'temperature',
 'wind_speed',
 'is_weekend',
 'gas_price',
 'coal_price',
 'price_lag_24h',
 'price_lag_168h',
 'price_rolling_24h',
 'price_rolling_168h',
 'co2_price',
 'is_holiday',
 'is_hol_or_week',
 'total_generation',
 'net_export',
 'coal_generation',
 'gas_generation',
 'nuclear_generation',
 'actual_wind_offshore',
 'actual_wind_onshore',
 'actual_solar',
 'actual_load']

In [3]:
# Cyclic transformation
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Check
print("New columns:")
new_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
print(df[new_cols].describe().round(3))

New columns:
        hour_sin   hour_cos    dow_sin    dow_cos  month_sin  month_cos
count  62728.000  62728.000  62728.000  62728.000  62728.000  62728.000
mean      -0.000     -0.000      0.001     -0.000      0.011      0.011
std        0.707      0.707      0.707      0.707      0.707      0.707
min       -1.000     -1.000     -0.975     -0.901     -1.000     -1.000
25%       -0.707     -0.707     -0.782     -0.901     -0.500     -0.500
50%        0.000     -0.000      0.000     -0.223      0.000      0.000
75%        0.707      0.707      0.782      0.623      0.866      0.866
max        1.000      1.000      0.975      1.000      1.000      1.000


In [4]:
# Lag features for fuel prices
df['gas_price_lag_24h'] = df['gas_price'].shift(24)
df['gas_price_lag_168h'] = df['gas_price'].shift(168)

df['coal_price_lag_24h'] = df['coal_price'].shift(24)
df['coal_price_lag_168h'] = df['coal_price'].shift(168)

df['co2_price_lag_24h'] = df['co2_price'].shift(24)

# Sanity check — first 170 rows should have NaN for 168h lags
print("NaNs from fuel lags:")
lag_cols = ['gas_price_lag_24h', 'gas_price_lag_168h',
            'coal_price_lag_24h', 'coal_price_lag_168h',
            'co2_price_lag_24h']
print(df[lag_cols].isnull().sum())

NaNs from fuel lags:
gas_price_lag_24h       24
gas_price_lag_168h     168
coal_price_lag_24h      24
coal_price_lag_168h    168
co2_price_lag_24h       24
dtype: int64


In [5]:
# Checkpoint: verify columns after cyclic + fuel lag features (should be ~41)
df.columns.tolist()

['timestamp',
 'price',
 'load',
 'wind_offshore',
 'wind_onshore',
 'solar',
 'hour',
 'day_of_week',
 'month',
 'temperature',
 'wind_speed',
 'is_weekend',
 'gas_price',
 'coal_price',
 'price_lag_24h',
 'price_lag_168h',
 'price_rolling_24h',
 'price_rolling_168h',
 'co2_price',
 'is_holiday',
 'is_hol_or_week',
 'total_generation',
 'net_export',
 'coal_generation',
 'gas_generation',
 'nuclear_generation',
 'actual_wind_offshore',
 'actual_wind_onshore',
 'actual_solar',
 'actual_load',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'gas_price_lag_24h',
 'gas_price_lag_168h',
 'coal_price_lag_24h',
 'coal_price_lag_168h',
 'co2_price_lag_24h']

In [6]:
# Renewable share — ratio of renewables to total generation
# More informative than absolute MW values
# NOTE: was previously named 'predicted_renewable_share' — renamed to 'renewable_share'
# for consistency with the rest of the feature set
df['renewable_share'] = (
    df['wind_onshore'] + df['wind_offshore'] + df['solar']
) / df['load'].replace(0, np.nan)

# Fuel cost index — weighted combo of gas and coal
# Gas is the strongest single predictor from EDA
df['fuel_cost_index'] = df['gas_price'] * 0.6 + df['coal_price'] * 0.4

# Dispatchable generation — thermal + nuclear (non-renewable, price-setting)
df['dispatchable_gen'] = (
    df['coal_generation'] + df['gas_generation'] + df['nuclear_generation']
)

# Demand-supply balance — load minus total generation
# Negative = oversupply (pushes price down), positive = undersupply
df['demand_supply_gap'] = df['load'] - df['total_generation']

# Sanity check
check_cols = ['renewable_share', 'fuel_cost_index',
              'dispatchable_gen', 'demand_supply_gap']
print(df[check_cols].describe().round(2))

       renewable_share  fuel_cost_index  dispatchable_gen  demand_supply_gap
count         62728.00         62728.00          62728.00           62728.00
mean              0.39            75.87          24956.94            -651.71
std               0.23            57.27          10403.24            7389.39
min               0.01            17.59           4158.75          -23374.00
25%               0.20            36.35          16982.19           -6006.12
50%               0.35            61.80          24708.71            -553.75
75%               0.54            78.63          32196.06            4875.03
max               1.30           349.46          57427.75           20980.00


In [7]:
# Peak hour flag (morning 7-10, evening 17-21)
df['is_peak_hour'] = df['hour'].isin(range(7, 11)) | df['hour'].isin(range(17, 22))
df['is_peak_hour'] = df['is_peak_hour'].astype(int)

# Wind suppression during peak — high wind during peak hours
# This is when wind has biggest price impact (merit order effect)
df['wind_x_peak'] = (
    (df['wind_onshore'] + df['wind_offshore']) * df['is_peak_hour']
)

# Gas price pressure during peak demand
df['gas_x_peak'] = df['gas_price'] * df['is_peak_hour']

# Solar suppression — solar during high demand hours
df['solar_x_demand'] = df['solar'] * df['load']

# Renewable share during peak — captures merit order effect
df['renewable_share_x_peak'] = df['renewable_share'] * df['is_peak_hour']

# Sanity check
interact_cols = ['is_peak_hour', 'wind_x_peak', 'gas_x_peak',
                 'solar_x_demand', 'renewable_share_x_peak']
print(df[interact_cols].describe().round(2))
print(f"\nPeak hours: {df['is_peak_hour'].sum():,} rows ({df['is_peak_hour'].mean()*100:.1f}%)")

       is_peak_hour  wind_x_peak  gas_x_peak  solar_x_demand   
count      62728.00     62728.00    62728.00    6.272800e+04  \
mean           0.38      5530.37       16.84    3.611918e+08   
std            0.48      9768.79       34.72    5.625654e+08   
min            0.00         0.00        0.00    0.000000e+00   
25%            0.00         0.00        0.00    0.000000e+00   
50%            0.00         0.00        0.00    1.151225e+07   
75%            1.00      7630.31       24.76    5.707494e+08   
max            1.00     51550.50      339.20    3.067909e+09   

       renewable_share_x_peak  
count                62728.00  
mean                     0.13  
std                      0.21  
min                      0.00  
25%                      0.00  
50%                      0.00  
75%                      0.23  
max                      1.16  

Peak hours: 23,526 rows (37.5%)


In [8]:
# Energy crisis period flag (2021-2023 gas/energy crisis)
df['is_crisis_period'] = (
    (df['timestamp'] >= '2021-01-01') &
    (df['timestamp'] <= '2023-12-31')
).astype(int)

# High price regime — above 75th percentile historically
price_75 = df['price'].quantile(0.75)
df['is_high_price_regime'] = (df['price'] > price_75).astype(int)

# Negative price flag
df['is_negative_price'] = (df['price'] < 0).astype(int)

# Year feature — captures long-term trend (energy transition)
df['year'] = df['timestamp'].dt.year

# Sanity check
print(f"Crisis period rows:    {df['is_crisis_period'].sum():,} ({df['is_crisis_period'].mean()*100:.1f}%)")
print(f"High price regime:     {df['is_high_price_regime'].sum():,} ({df['is_high_price_regime'].mean()*100:.1f}%)")
print(f"Negative price rows:   {df['is_negative_price'].sum():,} ({df['is_negative_price'].mean()*100:.1f}%)")
print(f"Years in dataset:      {sorted(df['year'].unique())}")

Crisis period rows:    26,254 (41.9%)
High price regime:     15,681 (25.0%)
Negative price rows:   2,037 (3.2%)
Years in dataset:      [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [9]:
# --- Additional derived features ---
# NOTE: These were originally in a later cell (after EXCLUDE_COLS).
# Moved here so all features exist before the single combined NaN drop below.

# Residual load — load not met by renewables (main driver of gas dispatch)
df['residual_load'] = df['load'] - (df['wind_onshore'] + df['wind_offshore'] + df['solar'])

# Load ramp — hour-over-hour change in demand
df['load_ramp'] = df['load'].diff(1)

# Renewable ramp — hour-over-hour change in renewable output
_res = df['wind_onshore'] + df['wind_offshore'] + df['solar']
df['renewable_ramp'] = _res.diff(1)

# Price volatility — 24h rolling std (backward-looking, no leakage)
df['price_volatility_24h'] = df['price'].rolling(24).std()

# Wind forecast total + 24h delta
df['total_wind_forecast'] = df['wind_onshore'] + df['wind_offshore']
df['delta_wind_forecast'] = df['total_wind_forecast'] - df['total_wind_forecast'].shift(24)

# --- Combined NaN drop ---
# Sources of NaNs:
#   price_lag_168h / coal_price_lag_168h / gas_price_lag_168h  → 168 rows
#   delta_wind_forecast                                         →  24 rows
#   price_volatility_24h                                        →  23 rows
#   load_ramp / renewable_ramp                                  →   1 row
# Dropping once here instead of two separate drops.
nan_cols = df.columns[df.isnull().any()].tolist()
print("NaNs per column before drop:")
print(df[nan_cols].isnull().sum().sort_values(ascending=False))

df = df.dropna().reset_index(drop=True)

print(f"\nShape after drop: {df.shape}")
print(f"Remaining NaNs:   {df.isnull().sum().sum()}")
print(f"Date range:       {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")

NaNs per column before drop:
gas_price_lag_168h      168
coal_price_lag_168h     168
gas_price_lag_24h        24
coal_price_lag_24h       24
co2_price_lag_24h        24
delta_wind_forecast      24
price_volatility_24h     23
load_ramp                 1
renewable_ramp            1
dtype: int64

Shape after drop: (62560, 60)
Remaining NaNs:   0
Date range:       2019-01-15 → 2026-03-05


In [10]:
# EXCLUDE_COLS: columns to keep in the saved CSV but NOT feed into the model.
# - timestamp: index/identifier, not a feature
# - price: target variable
# - is_high_price_regime / is_negative_price: derived from the target — would leak if used as features
# - actual_*: actual (post-hoc) values, not available at forecast time
EXCLUDE_COLS = [
    'timestamp',
    'price',
    'is_high_price_regime',
    'is_negative_price',
    'actual_wind_offshore',
    'actual_wind_onshore',
    'actual_solar',
    'actual_load',
]

feature_cols = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f"Total columns:   {df.shape[1]}")
print(f"Feature columns: {len(feature_cols)}")
print(f"Excluded:        {len(EXCLUDE_COLS)}")
print(f"\nFeatures going into model:")
for col in feature_cols:
    print(f"  {col}")

Total columns:   60
Feature columns: 52
Excluded:        8

Features going into model:
  load
  wind_offshore
  wind_onshore
  solar
  hour
  day_of_week
  month
  temperature
  wind_speed
  is_weekend
  gas_price
  coal_price
  price_lag_24h
  price_lag_168h
  price_rolling_24h
  price_rolling_168h
  co2_price
  is_holiday
  is_hol_or_week
  total_generation
  net_export
  coal_generation
  gas_generation
  nuclear_generation
  hour_sin
  hour_cos
  dow_sin
  dow_cos
  month_sin
  month_cos
  gas_price_lag_24h
  gas_price_lag_168h
  coal_price_lag_24h
  coal_price_lag_168h
  co2_price_lag_24h
  renewable_share
  fuel_cost_index
  dispatchable_gen
  demand_supply_gap
  is_peak_hour
  wind_x_peak
  gas_x_peak
  solar_x_demand
  renewable_share_x_peak
  is_crisis_period
  year
  residual_load
  load_ramp
  renewable_ramp
  price_volatility_24h
  total_wind_forecast
  delta_wind_forecast


In [11]:
# NOTE: residual_load, load_ramp, renewable_ramp, price_volatility_24h,
# total_wind_forecast, delta_wind_forecast were moved to the NaN-handling cell above.
# Reason: all features need to exist before the single combined NaN drop.

In [12]:
df.head()

,timestamp,price,load,wind_offshore,wind_onshore,solar,hour,day_of_week,month,temperature,...,is_crisis_period,is_high_price_regime,is_negative_price,year,residual_load,load_ramp,renewable_ramp,price_volatility_24h,total_wind_forecast,delta_wind_forecast
0,2019-01-15 00:00:00,35.18,54057.25,4999.00,24868.25,0.0,0,1,1,1.2875,...,0,0,0,2019,24190.00,-1161.00,-1270.50,23.024083,29867.25,-12156.00
1,2019-01-15 01:00:00,35.82,52364.25,4923.25,25149.75,0.0,1,1,1,1.2000,...,0,0,0,2019,22291.25,-1693.00,205.75,20.814584,30073.00,-11445.50
2,2019-01-15 02:00:00,34.99,51553.50,4883.25,25871.00,0.0,2,1,1,1.2625,...,0,0,0,2019,20799.25,-810.75,681.25,17.737597,30754.25,-10143.50
3,2019-01-15 03:00:00,34.56,51818.50,4831.00,26568.25,0.0,3,1,1,1.5000,...,0,0,0,2019,20419.25,265.00,645.00,14.333765,31399.25,-8869.25
4,2019-01-15 04:00:00,30.80,53052.00,4777.75,27191.75,0.0,4,1,1,1.5750,...,0,0,0,2019,21082.50,1233.50,570.25,10.882730,31969.50,-7537.00


In [13]:
#checking the data

In [14]:
# NOTE: NaN check moved to the combined NaN drop cell above (after 'is_crisis_period' block).
# No separate check needed here.

In [15]:
# NOTE: Second NaN drop merged into the combined NaN drop cell above.
# No separate drop needed here.

In [16]:
df.to_csv('../data/df_features.csv', index=False)
print(f"Saved: df_features.csv  ({df.shape[0]:,} rows, {df.shape[1]} cols)")

Saved: df_features.csv  (62,560 rows, 60 cols)


In [17]:
# MODELING

In [18]:
len(df.columns.tolist())

60

In [19]:
df.columns.tolist()

['timestamp',
 'price',
 'load',
 'wind_offshore',
 'wind_onshore',
 'solar',
 'hour',
 'day_of_week',
 'month',
 'temperature',
 'wind_speed',
 'is_weekend',
 'gas_price',
 'coal_price',
 'price_lag_24h',
 'price_lag_168h',
 'price_rolling_24h',
 'price_rolling_168h',
 'co2_price',
 'is_holiday',
 'is_hol_or_week',
 'total_generation',
 'net_export',
 'coal_generation',
 'gas_generation',
 'nuclear_generation',
 'actual_wind_offshore',
 'actual_wind_onshore',
 'actual_solar',
 'actual_load',
 'hour_sin',
 'hour_cos',
 'dow_sin',
 'dow_cos',
 'month_sin',
 'month_cos',
 'gas_price_lag_24h',
 'gas_price_lag_168h',
 'coal_price_lag_24h',
 'coal_price_lag_168h',
 'co2_price_lag_24h',
 'renewable_share',
 'fuel_cost_index',
 'dispatchable_gen',
 'demand_supply_gap',
 'is_peak_hour',
 'wind_x_peak',
 'gas_x_peak',
 'solar_x_demand',
 'renewable_share_x_peak',
 'is_crisis_period',
 'is_high_price_regime',
 'is_negative_price',
 'year',
 'residual_load',
 'load_ramp',
 'renewable_ramp',
 'pri

In [20]:
print(df['timestamp'].dtype)
print(df['timestamp'].head(3))

datetime64[ns]
0   2019-01-15 00:00:00
1   2019-01-15 01:00:00
2   2019-01-15 02:00:00
Name: timestamp, dtype: datetime64[ns]


In [21]:
hourly_counts = df.groupby(df['timestamp'].dt.date)['timestamp'].count()
bad_days = hourly_counts[hourly_counts != 24]
print(bad_days)

timestamp
2019-03-31    23
2020-03-29    23
2021-03-28    23
2022-03-27    23
2023-03-26    23
2024-03-31    23
2025-03-30    23
2026-03-05    23
Name: timestamp, dtype: int64
